# 🏛️ Hampi Revived — 3D Reconstruction Pipeline
> *Reconstructing the ruins of the Vijayanagara Empire (1336–1646 CE) using Computer Vision & Data Science. Stone meets Silicon.*

---

## Pipeline Overview

| Stage | Method | Output |
|-------|--------|--------|
| 1. Data Ingestion | Wikimedia Commons API | Raw + preprocessed images |
| 2. Preprocessing | CLAHE · Denoise · Sharpen | Enhanced images |
| 3. Feature Extraction | SIFT + FLANN | Keypoints, descriptors, matches |
| 4. Structure from Motion | Essential Matrix + RANSAC | Sparse 3D point cloud |
| 5. Dense Reconstruction | SGBM Stereo Depth | Dense coloured point cloud |
| 6. Mesh Reconstruction | Poisson Surface (Open3D) | 3D mesh (PLY / OBJ) |
| 7. Visualisation | Matplotlib + Plotly | Static PNGs + Interactive HTML |
| 8. Groq AI | llama-3.2-11b-vision-preview | Archaeological site report |


In [ ]:
# Environment & path setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # run from repo root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')
import cv2
import yaml
from IPython.display import display, Image, HTML

# Load config
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# Load .env
from pathlib import Path
if Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

print('✅ Environment ready')
print(f"Project: {cfg['project']['name']}")
print(f"Python:  {sys.version.split()[0]}")
print(f"OpenCV:  {cv2.__version__}")
import numpy; print(f"NumPy:   {numpy.__version__}")

---
## Stage 1 — Data Ingestion 📷
Download public-domain photographs of Hampi from Wikimedia Commons.
Falls back to procedurally-generated synthetic stone-ruin scenes if offline.

In [ ]:
from src.data_ingestion import download_images, load_images, dataset_stats

image_paths = download_images(
    save_dir=cfg['data']['raw_dir'],
    max_images=cfg['data']['max_images'],
    target_size=tuple(cfg['data']['target_size']),
)
print(f'\n📁 {len(image_paths)} images ready')
for p in image_paths:
    print(f'   {p}')

In [ ]:
target_size = tuple(cfg['data']['target_size'])
images_raw = load_images(image_paths, target_size)
stats = dataset_stats(images_raw)

print('Dataset Statistics')
print('─' * 40)
for k, v in stats.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# Show image grid
from src.visualization import plot_image_grid

grid_path = plot_image_grid(
    images_raw[:8],
    title='Hampi Dataset — Input Images',
    out_path='outputs/visualizations/image_grid.png',
)
display(Image(grid_path))

---
## Stage 2 — Preprocessing 🔬
**CLAHE** (Contrast Limited Adaptive Histogram Equalisation) boosts carved-stone detail.
**Non-local means denoising** removes sensor noise. **Unsharp masking** sharpens edges.

In [ ]:
from src.preprocessing import preprocess_batch, estimate_blur, estimate_exposure

images, quality = preprocess_batch(images_raw, denoise_imgs=True)

print(f'Preprocessed {len(images)} images')
print('\nQuality Report:')
print(f'{"Idx":>4}  {"Blur":>8}  {"Brightness":>10}  {"Exposure"}')
print('─' * 45)
for q in quality:
    warn = '  ⚠️  ' if q.get('warning') else ''
    print(f"{q['idx']:>4}  {q['blur_score']:>8.1f}  {q['brightness']:>10.1f}  {q['exposure']:12s}{warn}")

In [ ]:
# Side-by-side comparison: raw vs preprocessed
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Raw vs Preprocessed — Hampi Images', fontsize=14, fontweight='bold')

for i in range(min(4, len(images))):
    axes[0, i].imshow(cv2.cvtColor(images_raw[i], cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'Raw #{i}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f'Preprocessed #{i}\nblur={quality[i]["blur_score"]:.0f}', fontsize=9)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('outputs/visualizations/preprocess_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Preprocessing comparison saved')

---
## Stage 3 — Feature Extraction 🔑
**SIFT** (Scale-Invariant Feature Transform) detects stable keypoints in the granite carvings.
**FLANN** (Fast Library for Approximate Nearest Neighbours) matches descriptors across image pairs using Lowe's ratio test.

In [ ]:
from src.feature_extraction import (
    detect_and_describe, match_all_pairs,
    save_keypoints_plot, save_matches_plot,
    keypoint_stats, plot_keypoint_distribution
)

all_kps, all_descs = detect_and_describe(
    images,
    detector=cfg['features']['detector'],
    max_keypoints=cfg['features']['max_keypoints'],
)

# Update quality with keypoint counts
for i, q in enumerate(quality):
    q['n_keypoints'] = len(all_kps[i]) if i < len(all_kps) else 0

print('Keypoints per image:')
for i, kps in enumerate(all_kps):
    print(f'  Image {i}: {len(kps):,} keypoints')

In [ ]:
matches_dict = match_all_pairs(
    all_descs,
    ratio=cfg['features']['match_ratio_threshold'],
    min_matches=cfg['features']['min_matches_for_pair'],
)

feat_stats = keypoint_stats(all_kps, matches_dict)
print('Feature Statistics:')
import json
print(json.dumps(feat_stats, indent=2))

In [ ]:
# Keypoint distribution chart
dist_path = plot_keypoint_distribution(all_kps, out_path='outputs/features/keypoint_distribution.png')
display(Image(dist_path))

In [ ]:
# Match matrix heatmap
from src.visualization import plot_match_matrix
matrix_path = plot_match_matrix(len(images), matches_dict, out_path='outputs/visualizations/match_matrix.png')
display(Image(matrix_path))

In [ ]:
# Visualise keypoints on first image
kp_paths = save_keypoints_plot(images, all_kps, 'outputs/features', n_show=4)
match_paths = save_matches_plot(images, all_kps, matches_dict, 'outputs/features', max_pairs=3)

if kp_paths:
    print('Keypoint overlays:')
    fig, axes = plt.subplots(1, min(4, len(kp_paths)), figsize=(16, 4))
    if len(kp_paths) == 1: axes = [axes]
    for ax, p in zip(axes, kp_paths):
        img_kp = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        ax.imshow(img_kp); ax.axis('off')
    plt.suptitle('SIFT Keypoints on Hampi Images', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Show feature matches
if match_paths:
    print(f'Top feature matches ({len(match_paths)} pairs):')
    for p in match_paths[:2]:
        img_match = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(14, 5))
        plt.imshow(img_match)
        plt.axis('off')
        plt.title(f'SIFT Matches — {Path(p).stem}', fontsize=12)
        plt.tight_layout()
        plt.show()

---
## Stage 4 — Structure from Motion (SfM) 📐
For each image pair:
1. Extract matched point coordinates
2. Estimate **Essential Matrix** via RANSAC (1-pixel threshold)
3. **Recover camera pose** (R, t)
4. **Triangulate** 3D points
5. Accumulate into sparse point cloud

In [ ]:
from src.sfm import run_sfm, estimate_intrinsics

K = estimate_intrinsics(images[0].shape, cfg['sfm']['focal_length_factor'])
print('Camera Intrinsics K:')
print(K)
print(f'\nFocal length: {K[0,0]:.1f}px  (image {images[0].shape[1]}×{images[0].shape[0]})')

In [ ]:
pts3d_sparse, colors_sparse, camera_poses, cam_centres = run_sfm(
    images=images,
    all_kps=all_kps,
    matches_dict=matches_dict,
    img_shape=images[0].shape,
    focal_factor=cfg['sfm']['focal_length_factor'],
    ransac_threshold=cfg['sfm']['ransac_threshold'],
)

sfm_stats = {
    'n_images': len(images),
    'n_points': len(pts3d_sparse),
    'n_cameras': len(camera_poses),
    'n_pairs': len(matches_dict),
}

print('\n📊 SfM Results:')
print(f'  Sparse 3D points : {len(pts3d_sparse):,}')
print(f'  Cameras registered: {len(camera_poses)}')
print(f'  Camera centres   : {cam_centres.shape}')

# Save sparse cloud
np.save('outputs/point_clouds/sparse_pts.npy', pts3d_sparse)
np.save('outputs/point_clouds/sparse_cols.npy', colors_sparse)
print('  Sparse cloud saved to outputs/point_clouds/')

In [ ]:
# 3D scatter plot
from src.visualization import plot_sparse_cloud_3d
cloud_path = plot_sparse_cloud_3d(
    pts3d_sparse, colors_sparse, cam_centres,
    out_path='outputs/visualizations/sparse_cloud_3d.png',
)
if cloud_path:
    display(Image(cloud_path))

In [ ]:
# Interactive Plotly 3D (open in browser)
from src.visualization import plot_sparse_cloud_plotly
html_path = plot_sparse_cloud_plotly(
    pts3d_sparse, colors_sparse, cam_centres,
    out_path='outputs/visualizations/interactive_cloud.html',
)
if html_path:
    print(f'🌐 Open in browser: {os.path.abspath(html_path)}')
    # Inline mini preview
    display(HTML(f'<a href="{html_path}" target="_blank">▶ Open Interactive 3D Point Cloud</a>'))

---
## Stage 5 — Dense Reconstruction 🌊
**SGBM** (Semi-Global Block Matching) computes pixel-wise disparity between image pairs,
then depth is back-projected using camera intrinsics to get dense 3D point coverage.

In [ ]:
from src.dense_reconstruction import dense_reconstruct, voxel_downsample, compute_disparity

# Show disparity map for first pair
if len(images) >= 2:
    disp = compute_disparity(images[0], images[1])
    disp_vis = np.nan_to_num(disp, nan=0)
    disp_norm = (disp_vis / disp_vis.max() * 255).astype(np.uint8)
    disp_color = cv2.applyColorMap(disp_norm, cv2.COLORMAP_INFERNO)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
    axes[0].set_title('Image 0 (Left)', fontsize=11); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(images[1], cv2.COLOR_BGR2RGB))
    axes[1].set_title('Image 1 (Right)', fontsize=11); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(disp_color, cv2.COLOR_BGR2RGB))
    axes[2].set_title('SGBM Disparity Map\n(warm = near, cool = far)', fontsize=11); axes[2].axis('off')
    
    plt.suptitle('Semi-Global Block Matching — Stereo Depth', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/visualizations/disparity_map.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
pts3d_dense, colors_dense = dense_reconstruct(
    images=images,
    camera_poses=camera_poses,
    K=K,
    max_pairs=min(8, max(0, len(camera_poses) - 1)),
)

if len(pts3d_dense) > 0:
    pts3d_dense, colors_dense = voxel_downsample(
        pts3d_dense, colors_dense, voxel_size=cfg['reconstruction']['voxel_size']
    )
    print(f'✅ Dense cloud: {len(pts3d_dense):,} points (after voxel downsample)')
else:
    print('ℹ️  Dense reconstruction skipped (need ≥2 registered cameras)')

In [ ]:
# Merge sparse + dense
if len(pts3d_dense) > 0 and len(pts3d_sparse) > 0:
    pts_all = np.vstack([pts3d_sparse, pts3d_dense])
    cols_all = np.vstack([colors_sparse, colors_dense])
elif len(pts3d_sparse) > 0:
    pts_all, cols_all = pts3d_sparse, colors_sparse
else:
    pts_all, cols_all = pts3d_dense, colors_dense

print(f'Combined cloud: {len(pts_all):,} points')

# Top-down view
from src.visualization import plot_topdown
td_path = plot_topdown(pts_all, cols_all, out_path='outputs/visualizations/topdown_view.png')
if td_path:
    display(Image(td_path))

---
## Stage 6 — Mesh Reconstruction 🕸️
**Poisson surface reconstruction** fits a watertight surface to the oriented point cloud.
Low-density faces (artefacts) are removed; mesh is simplified via quadric decimation.

In [ ]:
from src.mesh import run_mesh_pipeline

mesh_result = run_mesh_pipeline(
    pts=pts_all,
    colors=cols_all,
    out_dir_pcd='outputs/point_clouds',
    out_dir_mesh='outputs/meshes',
    voxel_size=cfg['reconstruction']['voxel_size'],
    poisson_depth=cfg['reconstruction']['poisson_depth'],
)

print('Mesh Reconstruction Results:')
for k, v in mesh_result.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Visualise mesh stats
if mesh_result.get('n_triangles'):
    import plotly.graph_objects as go
    try:
        import open3d as o3d
        mesh = o3d.io.read_triangle_mesh(mesh_result['ply'])
        verts = np.asarray(mesh.vertices)
        tris = np.asarray(mesh.triangles)
        colors_mesh = np.asarray(mesh.vertex_colors) if mesh.has_vertex_colors() else None

        vc = (colors_mesh * 255).astype(int) if colors_mesh is not None else None
        colorscale = None
        intensity = None
        if vc is not None:
            intensity = (vc[:, 0] * 0.299 + vc[:, 1] * 0.587 + vc[:, 2] * 0.114)

        fig = go.Figure(data=[go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=tris[:, 0], j=tris[:, 1], k=tris[:, 2],
            intensity=intensity,
            colorscale='YlOrBr',
            opacity=0.9,
            name='Hampi Mesh',
        )])
        fig.update_layout(
            title=f'Hampi Revived — 3D Mesh ({mesh_result["n_triangles"]:,} triangles)',
            scene=dict(bgcolor='#0d1117'),
            paper_bgcolor='#0d1117',
            font=dict(color='white'),
        )
        fig.write_html('outputs/visualizations/mesh_interactive.html')
        fig.show()
        print(f'Interactive mesh: outputs/visualizations/mesh_interactive.html')
    except Exception as e:
        print(f'Mesh preview unavailable: {e}')
else:
    print('ℹ️  Mesh not available (need more 3D points or open3d installed)')

---
## Stage 7 — Quality Dashboard 🎨

In [ ]:
from src.visualization import plot_quality_dashboard

dash_path = plot_quality_dashboard(
    quality, sfm_stats,
    out_path='outputs/visualizations/quality_dashboard.png',
)
display(Image(dash_path))

---
## Stage 8 — Groq AI Archaeological Analysis 🤖
Using **llama-3.2-11b-vision-preview** for image analysis and
**llama-3.3-70b-versatile** for generating a structured site report.

> Set `GROQ_API_KEY` in `.env` to enable. Free tier at https://console.groq.com

In [ ]:
from src.groq_analysis import GroqArchaeologist, save_analyses

agent = GroqArchaeologist()

if agent.client is not None:
    print('✅ Groq connected — running archaeological analysis...')
    image_analyses = agent.analyse_batch(images_raw, n_images=min(4, len(images_raw)))
    
    print('\n=== Individual Image Analyses ===')
    for a in image_analyses:
        print(f'\n--- Image {a["idx"]} ---')
        print(a['analysis'][:500] + '...' if len(a['analysis']) > 500 else a['analysis'])
else:
    print('ℹ️  Groq not configured — add GROQ_API_KEY to .env')
    image_analyses = []

In [ ]:
if image_analyses:
    print('Generating full Archaeological Site Report...')
    site_report = agent.generate_site_report(image_analyses, sfm_stats)
    paths = save_analyses(image_analyses, site_report, 'outputs/reports')
    
    print('\n' + '═'*60)
    print('ARCHAEOLOGICAL SITE REPORT')
    print('═'*60)
    print(site_report)
    print(f'\n✅ Full report saved to: {paths["site_report"]}')
else:
    # Demo report without Groq
    demo = """# Hampi Revived — Archaeological Site Report (Demo)

## 1. Executive Summary
Hampi, a UNESCO World Heritage Site in Karnataka, India, was the capital of the
Vijayanagara Empire (1336–1646 CE). This reconstruction pipeline digitised the
site's architectural remains using multi-view stereo photogrammetry.

## 2. Identified Structures
- Virupaksha Temple (gopura, mandapa, inner sanctum)
- Stone Chariot (Garuda Mandapa, Vittala Temple complex)
- Elephant Stables (Islamic-influenced arched chambers)
- Lotus Mahal (Zenana Enclosure, Deccan–Vijayanagara hybrid)

## 3. Reconstruction Quality
Run with GROQ_API_KEY for full AI-generated report.
"""
    print(demo)

In [ ]:
# Ask Groq a specific question
if agent.client is not None:
    q = "What makes the Vittala Temple's stone chariot unique in Vijayanagara architecture?"
    print(f'Q: {q}\n')
    answer = agent.ask(q)
    print(f'A: {answer}')
else:
    print('Add GROQ_API_KEY to .env to use the AI Q&A feature')

---
## Summary — All Outputs 📁

In [ ]:
import json

summary = {
    'pipeline': 'Hampi Revived v1.0',
    'dataset': stats,
    'features': feat_stats,
    'sfm': sfm_stats,
    'dense_pts': len(pts3d_dense) if len(pts3d_dense) > 0 else 0,
    'total_pts': len(pts_all),
    'mesh': mesh_result,
}

print('┌─────────────────────────────────────────────────────────┐')
print('│             HAMPI REVIVED — Pipeline Summary            │')
print('├─────────────────────────────────────────────────────────┤')
print(f'│  Images processed    : {stats["n_images"]:>6}                           │')
print(f'│  Keypoints detected  : {feat_stats["keypoints"]["mean"]:>6.0f} avg/image                  │')
print(f'│  Connected pairs     : {feat_stats["connected_pairs"]:>6}                           │')
print(f'│  Sparse 3D points    : {len(pts3d_sparse):>6,}                           │')
print(f'│  Dense 3D points     : {(len(pts3d_dense) if len(pts3d_dense)>0 else 0):>6,}                           │')
print(f'│  Total 3D points     : {len(pts_all):>6,}                           │')
print(f'│  Cameras registered  : {len(camera_poses):>6}                           │')
if mesh_result.get("n_triangles"):
    print(f'│  Mesh triangles      : {mesh_result["n_triangles"]:>6,}                           │')
print('└─────────────────────────────────────────────────────────┘')

with open('outputs/reports/pipeline_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('\n✅ Full results saved to outputs/reports/pipeline_results.json')

In [ ]:
# List all generated outputs
import glob
print('Generated outputs:')
for pattern in ['outputs/**/*.png', 'outputs/**/*.html', 'outputs/**/*.ply', 'outputs/**/*.md']:
    for f in sorted(glob.glob(pattern, recursive=True)):
        size = os.path.getsize(f)
        print(f'  {f:55s}  {size/1024:>7.1f} KB')